# Invoke a Foundry agent via REST - multi-turn

Extends [`08-09-01-rest-single-shot.ipynb`](08-09-01-rest-single-shot.ipynb) by chaining two `POST /responses` calls together using `previous_response_id`. The result is the same continuation primitive that [`08-08-01-human-in-the-loop.ipynb`](../08-08-human-in-the-loop/08-08-01-human-in-the-loop.ipynb) uses to submit tool results back to the agent - the difference is what the second turn carries (a follow-up user message here, `function_call_output` items in HITL).

## 1. Setup

Identical to the single-shot notebook - same endpoint, same token scope, same agent. The only difference is the conversation now spans two requests.

In [ ]:
import json
import os
import subprocess
from pathlib import Path

import requests
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

AGENT_NAME = 'storytelling-agent'

repo_root = Path(subprocess.run(
    'git rev-parse --show-toplevel', shell=True, capture_output=True, text=True
).stdout.strip())
load_dotenv(repo_root / '.env', override=True)

endpoint = os.environ['ALPHA_FOUNDRY_PROJECT_ENDPOINT'].rstrip('/')
responses_url = f'{endpoint}/openai/v1/responses'

credential = DefaultAzureCredential()
access_token = credential.get_token('https://ai.azure.com/.default').token

headers = {
    'Authorization': f'Bearer {access_token}',
    'Content-Type': 'application/json',
}
agent_ref = {'name': AGENT_NAME, 'type': 'agent_reference'}


def output_text(result):
    """Aggregate the visible text from a raw Responses REST payload.

    The wire JSON has no top-level `output_text` key - that is a convenience the
    OpenAI SDK's typed Response synthesises by concatenating every output_text
    part across the structured `output` array. Over REST we do it ourselves.
    """
    return ''.join(
        part['text']
        for item in result.get('output', [])
        if item.get('type') == 'message'
        for part in item.get('content', [])
        if part.get('type') == 'output_text'
    )


print(f'Responses URL: {responses_url}')
print(f'Agent ref    : {agent_ref}')

## 2. Turn 1 - open the conversation

First POST is identical to the single-shot. The new step is capturing `result['id']` so the next turn can reference it.

In [ ]:
turn1_body = {
    'input': [
        {'role': 'user', 'content': 'Invent a one line story about an astronaut named Mira.'}
    ],
    'agent_reference': agent_ref,
}

r1 = requests.post(responses_url, headers=headers, json=turn1_body, timeout=60)
r1.raise_for_status()
turn1 = r1.json()

turn1_id = turn1['id']
print(f'Turn 1 response id: {turn1_id}')
print()
print('Turn 1 output:')
print(output_text(turn1))

## 3. Turn 2 - continue with previous_response_id

The second POST sends a new user message **and** sets `previous_response_id` to the first turn's id. The service rehydrates the prior conversation state on its side - the client does not have to resend turn 1's messages.

This is the same field name and semantics the SDK uses:

```python
openai_client.responses.create(
    input=...,
    previous_response_id=turn1.id,
    extra_body={'agent_reference': agent_ref},
)
```

In [ ]:
turn2_body = {
    'input': [
        {'role': 'user', 'content': 'Now tell me what happens next, in one line.'}
    ],
    'previous_response_id': turn1_id,
    'agent_reference': agent_ref,
}

r2 = requests.post(responses_url, headers=headers, json=turn2_body, timeout=60)
r2.raise_for_status()
turn2 = r2.json()

print(f'Turn 2 response id: {turn2["id"]}')
print(f'previous_response_id sent: {turn1_id}')
print()
print('Turn 2 output:')
print(output_text(turn2))

## 4. Reading the full conversation

Each turn returns a fresh `id`. The chain is implicit in the `previous_response_id` field - turn 2 points at turn 1, and a hypothetical turn 3 would point at turn 2. The server holds the conversation history; the client only needs the latest id.

In [ ]:
print('Conversation:')
print(f'  user  > {turn1_body["input"][0]["content"]}')
print(f'  agent > {output_text(turn1)}')
print(f'  user  > {turn2_body["input"][0]["content"]}')
print(f'  agent > {output_text(turn2)}')

## Summary

| Step | What changes from single-shot |
|------|-------------------------------|
| URL | Same - `{endpoint}/openai/v1/responses` |
| Auth | Same - bearer token, `https://ai.azure.com/.default` |
| Turn 1 body | Same as single-shot |
| Turn 2 body | Adds `'previous_response_id': turn1['id']` |
| State | Held server-side - client only needs the latest response id |

### Same primitive, different uses

`previous_response_id` is used for three distinct scenarios across this repo:

1. **Multi-turn conversation** (this notebook) - second turn carries a new user message.
2. **HITL tool-result submission** ([`08-08-01`](../08-08-human-in-the-loop/08-08-01-human-in-the-loop.ipynb)) - second turn carries `function_call_output` items keyed by the tool call ids from turn 1.
3. **MCP approval response** ([`08-05-02`](../08-05-contoso-pmo-mcp/08-05-02-contoso-pmo-agent-queries.ipynb) - implicit when `require_approval='always'`) - second turn carries an `mcp_approval_response` item.

The wire shape is the same in all three cases - only the items inside `input` differ.

Next: [`08-09-03-rest-streaming.ipynb`](08-09-03-rest-streaming.ipynb) shows the same endpoint with `stream: true` and Server-Sent Events parsing.